In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
torch.manual_seed(42)

In [ ]:
#I loaded this with AI
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Download and load the training data
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)

# Download and load the test data
test_data = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)


cpu


100%|██████████| 26.4M/26.4M [00:01<00:00, 13.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 213kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.99MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.2MB/s]


In [ ]:
# Extract images and labels from the PyTorch dataset
# Flatten the 28x28 images into 784 features
X_raw = train_data.data.numpy().reshape(-1, 28*28)
y_raw = train_data.targets.numpy()
X_test_raw = test_data.data.numpy().reshape(-1, 28*28)
y_test_raw = test_data.targets.numpy()
data = pd.DataFrame(X_test_raw)
data.insert(0, 'label', y_test_raw)
# Create a DataFrame
df = pd.DataFrame(X_raw)
# Insert the label as the first column to match your previous setup
df.insert(0, 'label', y_raw)
df


,label,0,1,2,3,4,5,6,7,8,...,774,775,776,777,778,779,780,781,782,783
0,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,...,119,114,130,76,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,33,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,0,0,0,0,0,0,0,0,0,0,...,66,54,50,5,0,1,0,0,0,0


In [ ]:
data

,label,0,1,2,3,4,5,6,7,8,...,774,775,776,777,778,779,780,781,782,783
0,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0,0,0,...,2,3,0,3,174,189,67,0,0,0
2,1,0,0,0,0,0,0,0,0,1,...,164,58,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,6,0,0,0,2,0,1,1,0,0,...,71,12,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9997,8,0,0,0,0,0,0,0,0,0,...,27,0,0,0,0,0,0,0,0,0
9998,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X_train = df.iloc[:,1:].values
y_train = df.iloc[:,0].values

X_test = data.iloc[:,1:].values
y_test = data.iloc[:,0].values

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, feature, label):
    self.feature = torch.tensor(feature, dtype = torch.float32)
    self.label = torch.tensor(label, dtype = torch.long)
  def __len__(self):
    return len(self.feature)
  def __getitem__(self,idx):
    return self.feature[idx], self.label[idx]

In [ ]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False, pin_memory = True)

In [ ]:
class MyNN(nn.Module):
  def __init__(self,num_feature):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_feature,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p = 0.3),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p = 0.3),
        nn.Linear(64,10)
    )
  def forward(self,x):
    return self.model(x)

In [ ]:
learning_rate = 0.1
epochs = 100

In [ ]:
model = MyNN(X_train.shape[1])
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = learning_rate, weight_decay = 1e-4)

In [ ]:
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_feature, batch_label in train_loader:
    batch_feature, batch_label =  batch_feature.to(device), batch_label.to(device)
    #forward
    optimizer.zero_grad()
    output = model.forward(batch_feature)

    # Loss calculation
    loss = criterion(output,batch_label)
    total_epoch_loss = total_epoch_loss + loss.item()
    #backward propogation

    loss.backward()

    #update these losses
    optimizer.step()
  avg = total_epoch_loss/len(train_loader)
  print(f"Epoch : {epoch + 1}/{epochs}\tLoss : {avg}")



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch : 1/100	Loss : 0.5963312254508336
Epoch : 2/100	Loss : 0.47604621710777284
Epoch : 3/100	Loss : 0.43789102178414663
Epoch : 4/100	Loss : 0.4190925233602524
Epoch : 5/100	Loss : 0.4026605217893918
Epoch : 6/100	Loss : 0.38963662945429484
Epoch : 7/100	Loss : 0.384269092853864
Epoch : 8/100	Loss : 0.37329054905970893
Epoch : 9/100	Loss : 0.3684948107520739
Epoch : 10/100	Loss : 0.35755913698871933
Epoch : 11/100	Loss : 0.35297055126428606
Epoch : 12/100	Loss : 0.35269551496505736
Epoch : 13/100	Loss : 0.34426276891827584
Epoch : 14/100	Loss : 0.3445620120505492
Epoch : 15/100	Loss : 0.33589330797990163
Epoch : 16/100	Loss : 0.3331909511486689
Epoch : 17/100	Loss : 0.33266587321956953
Epoch : 18/100	Loss : 0.3258260110259056
Epoch : 19/100	Loss : 0.3257596979121367
Epoch : 20/100	Loss : 0.3268877984642983
Epoch : 21/100	Loss : 0.3217553010503451
Epoch : 22/100	Loss : 0.31915865016380945
Epoch : 23/100	Loss : 0.321375741815567
Epoch : 24/100	Loss : 0.3138447121520837
Epoch : 25/100	L

In [ ]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
  for batch_feature, batch_label in test_loader:
    batch_feature, batch_label = batch_feature.to(device), batch_label.to(device)

    output = model(batch_feature)

    _, predicted = torch.max(output, 1)
    total = total + batch_label.shape[0]
    correct = correct + (predicted == batch_label).sum().item()
print(correct/total)


0.8887


In [ ]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
  for batch_feature, batch_label in train_loader:
    batch_feature, batch_label = batch_feature.to(device), batch_label.to(device)

    output = model(batch_feature)

    _, predicted = torch.max(output, 1)
    total = total + batch_label.shape[0]
    correct = correct + (predicted == batch_label).sum().item()
print(correct/total)

0.93955
